In [ ]:
!pip install transformers datasets torch librosa

In [ ]:
import torch
import numpy as np

from transformers import AutoFeatureExtractor, AutoModelForAudioClassification
from datasets import load_dataset, Audio, DatasetDict
from torch.utils.data import DataLoader

from scipy.special import softmax

import matplotlib.pyplot as plt

### Φόρτωση fine-tuned μοντέλου

In [ ]:
#Φόρτωση fine-tuned μοντέλου και feature extractor
repo_id = "VasilisAsim/data2vec-finetuned-TESS_full"
model_ft = AutoModelForAudioClassification.from_pretrained(repo_id)
feature_extractor = AutoFeatureExtractor.from_pretrained(repo_id)

device = "cuda" if torch.cuda.is_available() else "cpu"
model_ft = model_ft.to(device)
model_ft.eval()

In [ ]:
label2id = model_ft.config.label2id

### Φόρτωση dataset και προετοιμασία test split

In [ ]:
def load_tess_speaker_independent():
    """
    Συνάρτηση υπέυθηνη για τον speaker independent διαχωρισμό. Η ομιλητής OAF
    αντιστοιχίζεται στο σύνολο εκπαίδευσης και η YAF στο σύνολο δοκιμής
    """
    dataset = load_dataset("AbstractTTS/TESS", split="train")
    dataset = dataset.cast_column("audio", Audio(decode=False))   # για το filter

    def get_speaker(example):
        filename = example["audio"]["path"]
        if "OAF" in filename:
            return "OAF"
        elif "YAF" in filename:
            return "YAF"
        return "unknown"

    train_dataset = dataset.filter(lambda x: get_speaker(x) == "OAF")
    test_dataset = dataset.filter(lambda x: get_speaker(x) == "YAF")

    train_dataset = train_dataset.cast_column("audio", Audio(sampling_rate=16_000))
    test_dataset = test_dataset.cast_column("audio", Audio(sampling_rate=16_000))

    return DatasetDict({"train": train_dataset, "test": test_dataset})

In [ ]:
dataset = load_tess_speaker_independent()

In [ ]:
#Συνάρτηση προετοιμασίας του dataset: μετατροπή ήχου σε input_values μέσω του feature
#extractor και μετονομασία του "emotion" σε "labels" για συμβατότητα με το μοντέλο
def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_values"] = feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"]).input_values[0]
    batch["labels"] = batch["emotion"]
    return batch

#Εφαρμογή στο speaker-independent test split (ομιλητής YAF)
encoded_test = dataset["test"].map(prepare_dataset, remove_columns=dataset["test"].column_names)

### Δημιουργία DataLoader

In [ ]:
def collate_fn(features):
    """
    Μετατρέπει τα input_values του batch σε tensor και τα labels από strings
    σε ακέραια ids. Με batch_size=1 δεν γίνεται πραγματικό padding, ώστε οι
    πιθανότητες κάθε δείγματος να μην εξαρτώνται από τα υπόλοιπα του batch.
    """
    input_features = [{"input_values": f["input_values"]} for f in features]
    batch = feature_extractor.pad(input_features, padding=True, return_tensors="pt")
    batch["labels"] = torch.tensor(
        [label2id[f["labels"]] for f in features], dtype=torch.long
    )
    return batch

# batch_size=1: κάθε δείγμα περνά μόνο του από το μοντέλο, χωρίς padding, ώστε
# οι πιθανότητες να μην εξαρτώνται από τη σύνθεση του batch
test_loader = DataLoader(encoded_test, batch_size=1, collate_fn=collate_fn)

### Εξαγωγή logits στο test set

In [ ]:
all_logits, all_labels = [], []

with torch.no_grad():
    for batch in test_loader:
        input_values = batch["input_values"].to(device)
        labels = batch["labels"]

        outputs = model_ft(input_values=input_values)

        all_logits.append(outputs.logits.cpu().numpy())
        all_labels.append(labels.numpy())

logits = np.concatenate(all_logits, axis=0)
labels = np.concatenate(all_labels, axis=0)

### Συναρτήσεις μετρικών

In [ ]:
def negative_log_likelihood(probs, labels, eps=1e-12):
    """
    Μέσος όρος του -log(p_true) πάνω σε όλο το test set, όπου p_true η
    πιθανότητα που αποδίδει το μοντέλο στη σωστή κλάση κάθε δείγματος.
    """
    p_true = np.clip(probs[np.arange(len(labels)), labels], eps, 1.0)
    return -np.mean(np.log(p_true))


def brier_score(probs, labels_onehot):
    """
    Multiclass Brier score: μέσο τετραγωνικό σφάλμα ανάμεσα στο διάνυσμα
    πιθανοτήτων της πρόβλεψης και στο one-hot διάνυσμα της πραγματικής ετικέτας.
    """
    return np.mean(np.sum((probs - labels_onehot) ** 2, axis=1))


def predictive_entropy(probs, eps=1e-12):
    """
    Εντροπία της κατανομής πιθανοτήτων ανά δείγμα (φυσικός λογάριθμος).
    Μέγιστη τιμή ln(7) ≈ 1.95 για 7 κλάσεις (ομοιόμορφη κατανομή).
    """
    return -np.sum(probs * np.log(probs + eps), axis=1)

### Υπολογισμός πιθανοτήτων και μετρικών

In [ ]:
probs = softmax(logits, axis=1)
predictions = np.argmax(probs, axis=1)
confidences = np.max(probs, axis=1)
correct = (predictions == labels)

num_classes = probs.shape[1]
labels_onehot = np.eye(num_classes)[labels]

nll = negative_log_likelihood(probs, labels)
brier = brier_score(probs, labels_onehot)
entropy = predictive_entropy(probs)

In [ ]:
def expected_calibration_error(confidences, correct, n_bins=15):
    """
    Expected Calibration Error με bins ίσου πλάτους: το [0, 1] χωρίζεται σε
    n_bins διαστήματα βάσει του confidence κάθε πρόβλεψης, σε κάθε bin
    υπολογίζεται η απόλυτη διαφορά ανάμεσα σε accuracy και mean confidence
    """
    bin_edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    bin_info = []
    for i in range(n_bins):
        lo, hi = bin_edges[i], bin_edges[i + 1]
        # τελευταίο bin κλειστό και στα δύο άκρα
        in_bin = (confidences > lo) & (confidences <= hi) if i > 0 else (confidences >= lo) & (confidences <= hi)
        prop = np.mean(in_bin)
        if prop > 0:
            acc_bin = np.mean(correct[in_bin])
            conf_bin = np.mean(confidences[in_bin])
            ece += prop * abs(acc_bin - conf_bin)
            bin_info.append((lo, hi, acc_bin, conf_bin, prop))
    return ece, bin_info


def adaptive_calibration_error(confidences, correct, n_bins=15):
    """
    Adaptive Calibration Error με bins ίσου πλήθους δειγμάτων: τα δείγματα
    ταξινομούνται βάσει confidence και χωρίζονται σε ισομερή n_bins, και σε
    κάθε μέρος υπολογίζεται η ίδια διαφορά accuracy – confidence.
    """
    order = np.argsort(confidences)
    conf_sorted = confidences[order]
    correct_sorted = correct[order]
    splits = np.array_split(np.arange(len(conf_sorted)), n_bins)

    ace = 0.0
    bin_info = []
    for idx in splits:
        if len(idx) == 0:
            continue
        acc_bin = np.mean(correct_sorted[idx])
        conf_bin = np.mean(conf_sorted[idx])
        prop = len(idx) / len(conf_sorted)
        ace += prop * abs(acc_bin - conf_bin)
        bin_info.append((conf_bin, acc_bin, prop))
    return ace, bin_info

In [ ]:
ece, ece_bins = expected_calibration_error(confidences, correct, n_bins=15)
ace, ace_bins = adaptive_calibration_error(confidences, correct, n_bins=15)

### Οπτικοποίηση αποτελεσμάτων

In [ ]:
def plot_reliability_diagram(bin_info, title="Reliability Diagram"):
    """
    Reliability diagram: accuracy ανά bin συναρτήσει του mean confidence του bin.
    Η διαγώνιος αντιστοιχεί σε τέλεια βαθμονόμηση.
    """
    confs = [b[3] for b in bin_info]
    accs = [b[2] for b in bin_info]
    plt.figure(figsize=(5, 5))
    plt.bar(confs, accs, width=0.05, edgecolor="black", alpha=0.7, label="Model")
    plt.plot([0, 1], [0, 1], "k--", label="Perfect calibration")
    plt.xlabel("Confidence")
    plt.ylabel("Accuracy")
    plt.title(title)
    plt.legend()
    plt.xlim(0, 1)
    plt.ylim(0, 1)
    plt.show()

plot_reliability_diagram(ece_bins, title=f"Reliability Diagram (ECE={ece:.4f})")

In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(entropy[correct], bins=30, alpha=0.6, label="Σωστές προβλέψεις")
plt.hist(entropy[~correct], bins=30, alpha=0.6, label="Λάθος προβλέψεις")
plt.xlabel("Predictive Entropy")
plt.ylabel("Πλήθος δειγμάτων")
plt.title("Κατανομή Predictive Entropy")
plt.legend()
plt.show()

In [ ]:
#Confidence distribution: σωστές vs λάθος
plt.figure(figsize=(6, 4))
plt.hist(confidences[correct], bins=30, alpha=0.6, label="Σωστές προβλέψεις")
plt.hist(confidences[~correct], bins=30, alpha=0.6, label="Λάθος προβλέψεις")
plt.xlabel("Confidence (max probability)")
plt.ylabel("Πλήθος δειγμάτων")
plt.title("Κατανομή Confidence: Σωστά vs Λάθος")
plt.legend()
plt.show()

In [ ]:
#Σύνοψη
print(f"Accuracy: {correct.mean():.4f}")
print(f"NLL: {nll:.4f}")
print(f"Brier Score: {brier:.4f}")
print(f"ECE: {ece:.4f}")
print(f"ACE: {ace:.4f}")